# Prithvi-100M T=2 — Temporal Fusion for Wildfire Burn Scar Detection

**Environment**: Colab A100 (GPU required)

Architecture: Siamese Prithvi backbone (shared weights) + multi-scale TemporalFusionNeck + FPN decoder.

Each sample is a spatial pair: pre-fire image (Oct-Nov 2021) + post-fire image (Dec 2021-Feb 2022). The model learns to detect burn scars by directly comparing spectral change between the two dates.

---

| Section | Description |
|---|---|
| 1-3 | Install, Drive mount, Imports |
| 4-5 | Paths, Dataset |
| 6 | Architecture (TemporalFusionNeck + PrithviT2Model) |
| 7-8 | Phase 1: backbone frozen, 25 epochs |
| 9-10 | Phase 2: partial backbone unfreeze, 20 epochs |
| 11-12 | Threshold optimization + evaluation figure |

In [ ]:
# [1] Install
import subprocess, sys, os

needs = False
try:
    import numpy as np
    assert tuple(int(x) for x in np.__version__.split('.')[:2]) >= (2, 0)
    from terratorch.registry import BACKBONE_REGISTRY
    print(f'OK — numpy={np.__version__}, terratorch ready.')
except Exception as e:
    needs = True
    print(f'Installing ({e})...')

if needs:
    subprocess.run([sys.executable,'-m','pip','install','-q','numpy==2.0.2','--force-reinstall'], check=True)
    subprocess.run([sys.executable,'-m','pip','install','-q',
                    'terratorch','einops','timm','segmentation-models-pytorch'], check=True)
    print('Restarting...')
    os.kill(os.getpid(), 9)

In [ ]:
# [2] Drive mount
try:
    import google.colab; IN_COLAB = True
    from google.colab import drive; drive.mount('/content/drive')
except ImportError:
    IN_COLAB = False
print(f'IN_COLAB = {IN_COLAB}')

In [ ]:
# [3] Imports
import os, random, subprocess, warnings, json
from pathlib import Path
from tqdm import tqdm

import numpy as np
import torch
import torch.nn as nn
import torchvision.transforms.functional as TF
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.model_selection import train_test_split
import segmentation_models_pytorch as smp
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from terratorch.registry import BACKBONE_REGISTRY

warnings.filterwarnings('ignore')
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'torch={torch.__version__}  device={DEVICE}')
if torch.cuda.is_available(): print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# [4] Paths & constants
if IN_COLAB:
    BASE = Path('/content/drive/MyDrive/GeoAI/wildfire-spread')
else:
    BASE = Path('G:/Mon Drive/GeoAI/wildfire-spread')

# v1.5 checkpoint — backbone weights are transferred from here
CKPT_V15 = BASE / 'models/best_prithvi_burn_scar_wildfire.pth'
# T=2 checkpoint
CKPT_T2  = BASE / 'models/best_prithvi_burn_scar_t2.pth'
RESULTS  = BASE / 'results'
RESULTS.mkdir(parents=True, exist_ok=True)

if IN_COLAB:
    LOCAL = Path('/content/patches')
    LOCAL.mkdir(exist_ok=True)
    for subdir in ['images', 'images_prefire', 'masks_dnbr']:
        dst = LOCAL / subdir
        src = BASE / 'data' / 'patches' / subdir
        if not dst.exists():
            print(f'Copying {subdir}...')
            subprocess.run(['cp', '-r', str(src), str(LOCAL)], check=True)
    # Pair manifest
    import shutil
    shutil.copy(str(BASE/'data/patches/t2_pairs.json'), str(LOCAL/'t2_pairs.json'))
    IMG_DIR     = LOCAL / 'images'
    PRE_IMG_DIR = LOCAL / 'images_prefire'
    MASK_DIR    = LOCAL / 'masks_dnbr'
    PAIR_JSON   = LOCAL / 't2_pairs.json'
else:
    IMG_DIR     = BASE / 'data/patches/images'
    PRE_IMG_DIR = BASE / 'data/patches/images_prefire'
    MASK_DIR    = BASE / 'data/patches/masks_dnbr'
    PAIR_JSON   = BASE / 'data/patches/t2_pairs.json'

BAND_IDX     = [0, 1, 2, 4, 5, 6]   # B02,B03,B04,B8A,B11,B12
PRITHVI_MEAN = np.array([0.033349,0.057012,0.058897,0.232325,0.197285,0.119449], dtype=np.float32)
PRITHVI_STD  = np.array([0.022691,0.026808,0.040041,0.077917,0.087087,0.072420], dtype=np.float32)
IMG_SIZE = 224
T        = (256 - IMG_SIZE) // 2
BATCH    = 8
FIRE_W   = 5.0

with open(PAIR_JSON) as f:
    pair_names = json.load(f)
print(f'Valid T=2 pairs: {len(pair_names)}')

In [ ]:
# [5] Paired dataset class
class PairedBurnScarDataset(Dataset):
    def __init__(self, names, augment=False):
        self.names   = names
        self.augment = augment

    def __len__(self): return len(self.names)

    def __getitem__(self, idx):
        name = self.names[idx]
        raw_post = np.load(IMG_DIR     / name, mmap_mode='r').astype(np.float32)
        raw_pre  = np.load(PRE_IMG_DIR / name, mmap_mode='r').astype(np.float32)
        mask     = np.load(MASK_DIR    / name, mmap_mode='r')

        def normalize(raw):
            img = (raw[BAND_IDX] / 10000.0 - PRITHVI_MEAN[:, None, None]) / PRITHVI_STD[:, None, None]
            return img[:, T:T+IMG_SIZE, T:T+IMG_SIZE]

        post_t = torch.from_numpy(normalize(raw_post))
        pre_t  = torch.from_numpy(normalize(raw_pre))
        mask_c = mask[T:T+IMG_SIZE, T:T+IMG_SIZE].copy().astype(np.float32)
        mask_t = torch.from_numpy(mask_c).unsqueeze(0)

        if self.augment:
            # Identical transform applied to pre, post AND mask
            if torch.rand(1) > 0.5:
                post_t, pre_t, mask_t = TF.hflip(post_t), TF.hflip(pre_t), TF.hflip(mask_t)
            if torch.rand(1) > 0.5:
                post_t, pre_t, mask_t = TF.vflip(post_t), TF.vflip(pre_t), TF.vflip(mask_t)
            k = torch.randint(0, 4, (1,)).item()
            if k:
                post_t = TF.rotate(post_t, 90 * k)
                pre_t  = TF.rotate(pre_t,  90 * k)
                mask_t = TF.rotate(mask_t, 90 * k)

        return pre_t, post_t, mask_t

# Train / val split
print('Scanning fire flags...')
fire_flags = np.array([np.load(MASK_DIR/n, mmap_mode='r').max() > 0
                       for n in tqdm(pair_names)], dtype=np.int32)
train_names, val_names = train_test_split(
    pair_names, test_size=0.2, stratify=fire_flags, random_state=SEED)

fire_flags_train = np.array([np.load(MASK_DIR/n, mmap_mode='r').max() > 0
                              for n in tqdm(train_names, desc='train flags')], dtype=np.int32)
w = torch.tensor([FIRE_W if f else 1.0 for f in fire_flags_train], dtype=torch.float)
sampler = WeightedRandomSampler(w, len(w), replacement=True)

train_loader = DataLoader(
    PairedBurnScarDataset(train_names, augment=True),
    batch_size=BATCH, sampler=sampler, num_workers=2, pin_memory=True)
val_loader = DataLoader(
    PairedBurnScarDataset(val_names, augment=False),
    batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)
print(f'Train: {len(train_names)} | Val: {len(val_names)}')

---

## Architecture — Siamese Prithvi + TemporalFusionNeck

Two passes through the **same** Prithvi backbone (shared weights) produce two sets of transformer layer outputs, one for the pre-fire image and one for the post-fire image. The `TemporalFusionNeck` concatenates features at layers 2, 5, 8, 11 and fuses them via 1x1 convolutions before the top-down FPN pass. The FPN decoder is identical to v1.5.

Backbone weights are transferred from the v1.5 checkpoint; the temporal fusion convolutions are trained from scratch.

In [ ]:
# [6] Architecture
class TemporalFusionNeck(nn.Module):
    """Fuses pre-fire and post-fire feature maps at 4 transformer layers."""
    def __init__(self, n_side=14, d_model=768, fpn_ch=256):
        super().__init__()
        self.n_side = n_side
        # Per-layer: concat(pre, post) = 2*d_model -> d_model
        self.t_fuse = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(2 * d_model, d_model, 1, bias=False),
                nn.BatchNorm2d(d_model), nn.GELU()
            ) for _ in range(4)
        ])
        # Lateral: d_model -> fpn_ch
        self.lateral = nn.ModuleList([
            nn.Sequential(nn.Conv2d(d_model, fpn_ch, 1, bias=False),
                          nn.BatchNorm2d(fpn_ch), nn.GELU()) for _ in range(4)
        ])
        # Top-down
        self.td_conv = nn.ModuleList([
            nn.Sequential(nn.Conv2d(fpn_ch, fpn_ch, 3, padding=1, bias=False),
                          nn.BatchNorm2d(fpn_ch), nn.GELU()) for _ in range(3)
        ])

    def tok2map(self, tokens):
        B, L, D = tokens.shape
        return tokens.permute(0, 2, 1).reshape(B, D, self.n_side, self.n_side)

    def forward(self, pre_layers, post_layers):
        feats = []
        for i, idx in enumerate([2, 5, 8, 11]):
            pre_map  = self.tok2map(pre_layers[idx][:, 1:, :])
            post_map = self.tok2map(post_layers[idx][:, 1:, :])
            fused    = self.t_fuse[i](torch.cat([pre_map, post_map], dim=1))
            feats.append(self.lateral[i](fused))
        out = feats[3]
        out = self.td_conv[0](feats[2] + out)
        out = self.td_conv[1](feats[1] + out)
        out = self.td_conv[2](feats[0] + out)
        return out  # (B, 256, 14, 14)


class FPNDecoder(nn.Module):
    def __init__(self, in_ch=256):
        super().__init__()
        self.up = nn.Sequential(
            nn.ConvTranspose2d(in_ch, 256, 2, stride=2), nn.BatchNorm2d(256), nn.GELU(),
            nn.ConvTranspose2d(256,   128, 2, stride=2), nn.BatchNorm2d(128), nn.GELU(),
            nn.ConvTranspose2d(128,    64, 2, stride=2), nn.BatchNorm2d(64),  nn.GELU(),
            nn.ConvTranspose2d(64,     32, 2, stride=2), nn.BatchNorm2d(32),  nn.GELU(),
            nn.Conv2d(32, 1, 1),
        )
    def forward(self, x): return self.up(x)


class PrithviT2Model(nn.Module):
    """Siamese Prithvi backbone with temporal fusion neck."""
    def __init__(self):
        super().__init__()
        self.backbone = BACKBONE_REGISTRY.build('prithvi_eo_v1_100')
        self.neck     = TemporalFusionNeck(n_side=14, d_model=768, fpn_ch=256)
        self.decoder  = FPNDecoder(in_ch=256)

    def forward(self, pre, post):
        pre_layers  = self.backbone(pre.unsqueeze(2))
        post_layers = self.backbone(post.unsqueeze(2))
        return self.decoder(self.neck(pre_layers, post_layers))


dice_loss  = smp.losses.DiceLoss(mode='binary')
focal_loss = smp.losses.FocalLoss(mode='binary')

def criterion(logits, targets):
    return dice_loss(logits, targets) + focal_loss(logits, targets)

def fire_iou(logits, target, threshold=0.525):
    pred = (logits.sigmoid() > threshold).float()
    if target.sum() == 0 and pred.sum() == 0: return None
    tp = (pred * target).sum()
    fp = (pred * (1 - target)).sum()
    fn = ((1 - pred) * target).sum()
    return (tp / (tp + fp + fn + 1e-6)).item()

print('Architecture defined.')

---

## Phase 1 — Backbone Frozen (25 epochs)

Weight transfer from v1.5 checkpoint:
- `backbone`: exact weight match
- `neck.lateral` + `neck.td_conv`: transferred from v1.5 MultiScaleNeck
- `decoder`: exact weight match
- `neck.t_fuse`: **new**, trained from scratch

Learning rates: t_fuse=5e-4, lateral+td_conv+decoder=5e-5 (fine-tune pretrained)

In [ ]:
# [7] Phase 1 setup — load v1.5 weights
model = PrithviT2Model().to(DEVICE)

v15_state = torch.load(CKPT_V15, map_location=DEVICE)

# Transfer backbone weights
bb_state = {k[len('backbone.'):]: v for k, v in v15_state.items()
            if k.startswith('backbone.')}
model.backbone.load_state_dict(bb_state, strict=True)
print(f'Backbone: {len(bb_state)} tensors loaded from v1.5')

# Transfer neck.lateral and neck.td_conv from v1.5 MultiScaleNeck
neck_lateral = {k[len('neck.lateral.'):]: v for k, v in v15_state.items()
                if k.startswith('neck.lateral.')}
neck_td      = {k[len('neck.td_conv.'):]: v for k, v in v15_state.items()
                if k.startswith('neck.td_conv.')}
model.neck.lateral.load_state_dict(neck_lateral, strict=True)
model.neck.td_conv.load_state_dict(neck_td, strict=True)
print(f'Neck lateral: {len(neck_lateral)} tensors transferred')
print(f'Neck td_conv: {len(neck_td)} tensors transferred')

# Transfer decoder
dec_state = {k[len('decoder.'):]: v for k, v in v15_state.items()
             if k.startswith('decoder.')}
model.decoder.load_state_dict(dec_state, strict=True)
print(f'Decoder: {len(dec_state)} tensors transferred')
print('neck.t_fuse: initialized from scratch')

# Freeze backbone
for p in model.parameters(): p.requires_grad = False
for p in model.neck.t_fuse.parameters():   p.requires_grad = True
for p in model.neck.lateral.parameters():  p.requires_grad = True
for p in model.neck.td_conv.parameters():  p.requires_grad = True
for p in model.decoder.parameters():       p.requires_grad = True

NUM_EP_P1 = 25
optimizer_p1 = torch.optim.AdamW([
    {'params': model.neck.t_fuse.parameters(),   'lr': 5e-4},
    {'params': list(model.neck.lateral.parameters()) +
               list(model.neck.td_conv.parameters()) +
               list(model.decoder.parameters()),   'lr': 5e-5},
], weight_decay=1e-4)
scheduler_p1 = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer_p1, T_max=NUM_EP_P1, eta_min=1e-5)
print(f'Phase 1 ready: {NUM_EP_P1} epochs | backbone frozen')

In [ ]:
# [8] Phase 1 training loop
best_iou    = 0.0
history_p1  = {'train_loss': [], 'val_loss': [], 'val_iou': []}

for epoch in range(1, NUM_EP_P1 + 1):
    model.train(); model.backbone.eval()
    ep_loss = 0.0
    for pre, post, masks in tqdm(train_loader,
                                  desc=f'P1 {epoch:02d}/{NUM_EP_P1} [train]', leave=False):
        pre, post, masks = pre.to(DEVICE), post.to(DEVICE), masks.to(DEVICE)
        optimizer_p1.zero_grad()
        loss = criterion(model(pre, post), masks)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer_p1.step()
        ep_loss += loss.item()
    scheduler_p1.step()
    history_p1['train_loss'].append(ep_loss / len(train_loader))

    model.eval()
    v_loss, v_iou_list = 0.0, []
    with torch.no_grad():
        for pre, post, masks in tqdm(val_loader,
                                      desc=f'P1 {epoch:02d}/{NUM_EP_P1} [val]', leave=False):
            pre, post, masks = pre.to(DEVICE), post.to(DEVICE), masks.to(DEVICE)
            preds = model(pre, post)
            v_loss += criterion(preds, masks).item()
            iou = fire_iou(preds, masks)
            if iou is not None: v_iou_list.append(iou)
    history_p1['val_loss'].append(v_loss / len(val_loader))
    history_p1['val_iou'].append(np.mean(v_iou_list) if v_iou_list else 0.0)

    print(f'P1 Epoch {epoch:02d} | '
          f'Train: {history_p1["train_loss"][-1]:.4f} | '
          f'Val: {history_p1["val_loss"][-1]:.4f} | '
          f'IoU: {history_p1["val_iou"][-1]:.4f}')

    if history_p1['val_iou'][-1] > best_iou:
        best_iou = history_p1['val_iou'][-1]
        torch.save(model.state_dict(), CKPT_T2)
        print(f'  -> Saved (IoU: {best_iou:.4f})')

print(f'Phase 1 complete. Best IoU: {best_iou:.4f}')

---

## Phase 2 — Partial Backbone Unfreeze (20 epochs)

Last 2 transformer blocks + norm layer unfrozen. Differential LR: backbone=1e-5, t_fuse=5e-5, lateral+decoder=1e-5. Gradient clipping max_norm=1.0.

In [ ]:
# [9] Phase 2 setup
model.load_state_dict(torch.load(CKPT_T2, map_location=DEVICE))

for p in model.parameters(): p.requires_grad = False
for p in model.backbone.blocks[-2:].parameters(): p.requires_grad = True
for p in model.backbone.norm.parameters():        p.requires_grad = True
for p in model.neck.parameters():                 p.requires_grad = True
for p in model.decoder.parameters():              p.requires_grad = True

NUM_EP_P2 = 20
bb_params   = [p for p in model.backbone.parameters() if p.requires_grad]
fuse_params = list(model.neck.t_fuse.parameters())
rest_params = [p for p in
               list(model.neck.lateral.parameters()) +
               list(model.neck.td_conv.parameters()) +
               list(model.decoder.parameters()) if p.requires_grad]

optimizer_p2 = torch.optim.AdamW([
    {'params': bb_params,   'lr': 1e-5},
    {'params': fuse_params, 'lr': 5e-5},
    {'params': rest_params, 'lr': 1e-5},
], weight_decay=1e-4)
scheduler_p2 = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer_p2, T_max=NUM_EP_P2, eta_min=1e-6)
print(f'Phase 2 ready: {NUM_EP_P2} epochs | last 2 blocks + neck + decoder')

In [ ]:
# [10] Phase 2 training loop
history_p2 = {'train_loss': [], 'val_loss': [], 'val_iou': []}

for epoch in range(1, NUM_EP_P2 + 1):
    model.train()
    model.backbone.eval()
    model.backbone.blocks[-2].train()
    model.backbone.blocks[-1].train()
    model.backbone.norm.train()

    ep_loss = 0.0
    for pre, post, masks in tqdm(train_loader,
                                  desc=f'P2 {epoch:02d}/{NUM_EP_P2} [train]', leave=False):
        pre, post, masks = pre.to(DEVICE), post.to(DEVICE), masks.to(DEVICE)
        optimizer_p2.zero_grad()
        loss = criterion(model(pre, post), masks)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer_p2.step()
        ep_loss += loss.item()
    scheduler_p2.step()
    history_p2['train_loss'].append(ep_loss / len(train_loader))

    model.eval()
    v_loss, v_iou_list = 0.0, []
    with torch.no_grad():
        for pre, post, masks in tqdm(val_loader,
                                      desc=f'P2 {epoch:02d}/{NUM_EP_P2} [val]', leave=False):
            pre, post, masks = pre.to(DEVICE), post.to(DEVICE), masks.to(DEVICE)
            preds = model(pre, post)
            v_loss += criterion(preds, masks).item()
            iou = fire_iou(preds, masks)
            if iou is not None: v_iou_list.append(iou)
    history_p2['val_loss'].append(v_loss / len(val_loader))
    history_p2['val_iou'].append(np.mean(v_iou_list) if v_iou_list else 0.0)

    print(f'P2 Epoch {epoch:02d} | '
          f'Train: {history_p2["train_loss"][-1]:.4f} | '
          f'Val: {history_p2["val_loss"][-1]:.4f} | '
          f'IoU: {history_p2["val_iou"][-1]:.4f}')

    if history_p2['val_iou'][-1] > best_iou:
        best_iou = history_p2['val_iou'][-1]
        torch.save(model.state_dict(), CKPT_T2)
        print(f'  -> Saved (IoU: {best_iou:.4f})')

print(f'Training complete. Best IoU: {best_iou:.4f}')

---

## Evaluation

Threshold optimization and portfolio figure comparing T=1 (v1.5) and T=2.

In [ ]:
# [11] Threshold optimization
model.load_state_dict(torch.load(CKPT_T2, map_location=DEVICE))
model.eval()

all_probs, all_targets = [], []
with torch.no_grad():
    for name in tqdm(val_names, desc='Collecting probs'):
        raw_post = np.load(IMG_DIR     / name, mmap_mode='r').astype(np.float32)
        raw_pre  = np.load(PRE_IMG_DIR / name, mmap_mode='r').astype(np.float32)
        mask     = np.load(MASK_DIR    / name, mmap_mode='r')

        def norm(raw):
            img = (raw[BAND_IDX] / 10000.0 - PRITHVI_MEAN[:,None,None]) / PRITHVI_STD[:,None,None]
            return img[:, T:T+IMG_SIZE, T:T+IMG_SIZE]

        pre_t  = torch.from_numpy(norm(raw_pre)).unsqueeze(0).to(DEVICE)
        post_t = torch.from_numpy(norm(raw_post)).unsqueeze(0).to(DEVICE)
        mc     = mask[T:T+IMG_SIZE, T:T+IMG_SIZE]
        prob   = model(pre_t, post_t).sigmoid().squeeze().cpu().numpy()
        all_probs.append(prob.ravel())
        all_targets.append(mc.ravel())

all_probs   = np.concatenate(all_probs).astype(np.float32)
all_targets = np.concatenate(all_targets).astype(np.uint8)

thresholds = np.arange(0.30, 0.85, 0.025)
sweep = []
for t in thresholds:
    preds = (all_probs > t).astype(np.int32)
    tp = int(((preds==1)&(all_targets==1)).sum())
    fp = int(((preds==1)&(all_targets==0)).sum())
    fn = int(((preds==0)&(all_targets==1)).sum())
    iou  = tp/(tp+fp+fn+1e-6)
    prec = tp/(tp+fp+1e-6)
    rec  = tp/(tp+fn+1e-6)
    f1   = 2*prec*rec/(prec+rec+1e-6)
    sweep.append((t, iou, f1, prec, rec))
sweep = np.array(sweep)
bi    = sweep[:,1].argmax()
THRESHOLD_OPT = float(sweep[bi, 0])

print(f'Optimal threshold: {THRESHOLD_OPT:.3f}')
print(f'IoU={sweep[bi,1]:.4f}  F1={sweep[bi,2]:.4f}  '
      f'Prec={sweep[bi,3]:.4f}  Rec={sweep[bi,4]:.4f}')
print()
print('=== T=2 vs T=1 comparison ===')
print(f'v1.5 (T=1): IoU=0.5385  F1=0.7000  Prec=0.6934  Rec=0.7068')
print(f'T=2       : IoU={sweep[bi,1]:.4f}  F1={sweep[bi,2]:.4f}  '
      f'Prec={sweep[bi,3]:.4f}  Rec={sweep[bi,4]:.4f}')
delta_iou = sweep[bi,1] - 0.5385
print(f'Delta IoU : {delta_iou:+.4f} ({delta_iou/0.5385*100:+.1f}%)')

In [ ]:
# [12] Evaluation figure
all_samples = []
tp_all = fp_all = fn_all = 0
model.eval()
with torch.no_grad():
    for name in tqdm(val_names, desc='Evaluating'):
        raw_post = np.load(IMG_DIR     / name, mmap_mode='r').astype(np.float32)
        raw_pre  = np.load(PRE_IMG_DIR / name, mmap_mode='r').astype(np.float32)
        mask     = np.load(MASK_DIR    / name, mmap_mode='r')

        def norm(raw):
            img = (raw[BAND_IDX]/10000.0 - PRITHVI_MEAN[:,None,None]) / PRITHVI_STD[:,None,None]
            return img[:, T:T+IMG_SIZE, T:T+IMG_SIZE]

        pre_t  = torch.from_numpy(norm(raw_pre)).unsqueeze(0).to(DEVICE)
        post_t = torch.from_numpy(norm(raw_post)).unsqueeze(0).to(DEVICE)
        mc     = mask[T:T+IMG_SIZE, T:T+IMG_SIZE]
        prob   = model(pre_t, post_t).sigmoid().squeeze().cpu().numpy()
        pred   = (prob > THRESHOLD_OPT).astype(np.uint8)
        tp = int(((pred==1)&(mc==1)).sum())
        fp = int(((pred==1)&(mc==0)).sum())
        fn = int(((pred==0)&(mc==1)).sum())
        iou = tp/(tp+fp+fn+1e-6)
        tp_all+=tp; fp_all+=fp; fn_all+=fn
        rgb = raw_post[[2,1,0], T:T+IMG_SIZE, T:T+IMG_SIZE]
        rgb = (rgb/(np.percentile(rgb,98)+1e-6)*255).clip(0,255).astype(np.uint8).transpose(1,2,0)
        all_samples.append((iou, rgb, mc, pred, mc.sum()/(IMG_SIZE**2)))

g_iou  = tp_all/(tp_all+fp_all+fn_all+1e-6)
g_prec = tp_all/(tp_all+fp_all+1e-6)
g_rec  = tp_all/(tp_all+fn_all+1e-6)
g_f1   = 2*g_prec*g_rec/(g_prec+g_rec+1e-6)
print(f'Global: IoU={g_iou:.4f}  F1={g_f1:.4f}  Prec={g_prec:.4f}  Rec={g_rec:.4f}')

vis = [s for s in all_samples if 0.08 <= s[4] <= 0.60]
vis.sort(key=lambda x: x[0])
n = len(vis)
picks = [('BEST', vis[int(n*.97)]), ('MED #1', vis[int(n*.62)]),
         ('MED #2', vis[int(n*.38)]), ('WORST', vis[int(n*.03)])]

def overlay(mask, pred):
    rgba = np.zeros((*mask.shape, 4), dtype=np.float32)
    rgba[(pred==1)&(mask==1)] = [0.05,0.82,0.12,0.30]
    rgba[(pred==1)&(mask==0)] = [1.00,0.52,0.00,0.62]
    rgba[(pred==0)&(mask==1)] = [0.88,0.02,0.05,0.62]
    return rgba

fig = plt.figure(figsize=(15, 7.5), facecolor='white')
fig.suptitle('Wildfire Burn Scar Detection — Siamese Prithvi T=2 (Pre-fire + Post-fire)',
             fontsize=13, fontweight='bold', y=0.97)
fig.text(0.5, 0.92, f'T=2 IoU={g_iou:.4f}  F1={g_f1:.4f}  |  '
         f'vs v1.5 (T=1): IoU=0.5385  |  delta = {g_iou-0.5385:+.4f}',
         ha='center', fontsize=9, color='#333')
gs  = gridspec.GridSpec(1, 2, figure=fig, left=0.02, right=0.97,
                        top=0.89, bottom=0.09, wspace=0.05, width_ratios=[2.1, 1])
gs_p = gridspec.GridSpecFromSubplotSpec(2, 2, subplot_spec=gs[0], hspace=0.03, wspace=0.03)
lcol = {'BEST':'#097a27','MED #1':'#b86e00','MED #2':'#b86e00','WORST':'#a30000'}
for (label, s), (r, c) in zip(picks, [(0,0),(0,1),(1,0),(1,1)]):
    iou, rgb, mc, pred = s[0], s[1], s[2], s[3]
    ax = fig.add_subplot(gs_p[r, c])
    ax.imshow(rgb, interpolation='bilinear')
    ax.imshow(overlay(mc, pred), interpolation='none')
    ax.text(0.012, 0.985, f'{label}  |  Patch IoU = {iou:.3f}', transform=ax.transAxes,
            va='top', ha='left', fontsize=8.5, fontweight='bold', color='white',
            bbox=dict(facecolor=lcol[label], edgecolor='none', pad=2.5, alpha=0.88))
    ax.axis('off')

gs_r = gridspec.GridSpecFromSubplotSpec(2, 1, subplot_spec=gs[1], hspace=0.44)
ax_b = fig.add_subplot(gs_r[0])
ep_all  = list(range(1, len(history_p1['val_iou']) + len(history_p2['val_iou']) + 1))
iou_all = history_p1['val_iou'] + history_p2['val_iou']
ax_b.plot(ep_all, iou_all, color='#1aaa44', lw=1.5, label='T=2 val IoU')
ax_b.axhline(0.5385, color='steelblue', lw=1.2, ls='--', label='v1.5 (T=1) 0.5385')
ax_b.axvline(25, color='#888', ls=':', lw=1.2, label='Phase 1->2')
ax_b.set_title('Training convergence (45 epochs)', fontsize=8.5, fontweight='bold')
ax_b.set_xlabel('Epoch', fontsize=7.5); ax_b.tick_params(labelsize=7)
ax_b.grid(alpha=0.2); ax_b.legend(fontsize=7, frameon=False)

ax_m = fig.add_subplot(gs_r[1])
names  = ['Recall','Precision','F1','IoU']; values = [g_rec, g_prec, g_f1, g_iou]
bars   = ax_m.barh(names, values, color='#4a7fbf', height=0.48)
for bar, val in zip(bars, values):
    ax_m.text(val+0.012, bar.get_y()+bar.get_height()/2, f'{val:.3f}',
              va='center', fontsize=9.5, fontweight='bold', color='#111')
ax_m.set_xlim(0, 1.0)
ax_m.set_title(f'Global metrics  (n={len(val_names):,} pairs)', fontsize=8.5, fontweight='bold')
ax_m.tick_params(labelsize=8.5); ax_m.grid(axis='x', alpha=0.2)
ax_m.spines[['top','right','left']].set_visible(False)

fig.legend(handles=[
    mpatches.Patch(color=(0.05,0.82,0.12), label='True Positive'),
    mpatches.Patch(color=(1.00,0.52,0.00), label='False Positive'),
    mpatches.Patch(color=(0.88,0.02,0.05), label='False Negative'),
], loc='lower left', ncol=3, fontsize=8, frameon=False, bbox_to_anchor=(0.02, 0.005))
fig.text(0.5, 0.012,
         'Siamese Prithvi-EO-1.0-100M  |  Pre-fire (Oct-Nov 2021) + Post-fire (Dec 2021-Feb 2022)'
         '  |  Corrientes, Argentina  |  dNBR > 0.10 labels',
         ha='center', fontsize=7.5, color='#555')
plt.savefig(RESULTS / 'validation_overview_t2.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print(f'Saved: {RESULTS}/validation_overview_t2.png')